# Stage 4: Run Indexing, Chunking, & Vector Search Benchmark

Evaluates 5 Embedding Models x 5 Chunking Strategies across the canonical 1,034-page historical corpus.

In [ ]:
import os
import sys
from pathlib import Path

# Add code directory to path
for p in [Path("code"), Path("../code"), Path("extras/indexing-benchmarks/code")]:
    if p.is_dir():
        sys.path.insert(0, str(p.resolve()))
        break

from corpus import load_canonical_corpus
from queries import load_retrieval_queries
from runner import run_stage4_benchmark

In [ ]:
# Determine working and output directories
if Path("/kaggle/working").is_dir():
    OUT_DIR = Path("/kaggle/working/results")
    SEARCH_ROOT = Path("/kaggle/input")
else:
    OUT_DIR = Path("results")
    SEARCH_ROOT = Path(".")

In [ ]:
# Load and inspect dataset
pages = load_canonical_corpus(SEARCH_ROOT)
queries = load_retrieval_queries(SEARCH_ROOT)

print(f"Total Pages: {len(pages)} ({sum(1 for p in pages if p.word_count > 0)} non-empty)")
print(f"Total Words: {sum(p.word_count for p in pages):,}")
print(f"Total Grounded Queries: {len([q for q in queries if q.page_ids])}")

In [ ]:
# Run the complete 5x5 Factorial Benchmark
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Executing on: {device.upper()}")

report = run_stage4_benchmark(OUT_DIR, device=device, search_root=SEARCH_ROOT)
print(f"\nWinning Stack: {report['selection_decision']['winning_embedding_model']} + {report['selection_decision']['winning_chunking_strategy']}")